# Imports

In [ ]:
!pip install torch_geometric

In [3]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import math
import gc

import torch
import torch.nn as nn

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix
from scipy.special import expit # It is the optimized sigmoid of Scipy


In [4]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [5]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
#exp_manager = ExperimentManager()

# Load data

In [6]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Model

In [7]:
class EdgeMLP(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout, output_bias_init=None):
        super().__init__()

        input_total_dim = (2 * node_dim) + edge_dim

        # We use LayerNorm instead of BatchNorm1d
        # LayerNorm does not fail with batch_size=1
        self.net = nn.Sequential(
            nn.Linear(input_total_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.net[-1].bias.data.fill_(output_bias_init)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        edge_rep = torch.cat([x[src], x[dst], edge_attr], dim=1)
        return self.net(edge_rep)


# Functions


## Metrics

In [8]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [9]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## pos_weight

In [9]:
def check_class_balance(loader, name="Loader"):
    total_benign = 0
    total_attack = 0

    print(f"--- Analyzing {name} ---")
    for data in loader:
        # data.y is [1] or [N]
        y = data.y.view(-1)
        n_ones = y.sum().item()
        n_zeros = y.shape[0] - n_ones

        total_attack += n_ones
        total_benign += n_zeros

    print(f"Benign: {int(total_benign)}")
    print(f"Attack : {int(total_attack)}")

    if total_attack > 0:
        # POS_WEIGHT: (Negative / Positive)
        # Used to penalize errors in the minority class more severely
        ratio = total_benign / total_attack
        pos_weight_tensor = torch.tensor([ratio])

        # BIAS_INIT: log(Positives / Negatives)
        # Used to allow the network to "guess" the base probability from the start
        # Mathematically equivalent to: -math.log(ratio)
        bias_val = math.log(total_attack / total_benign)

        print(f"Ratio (pos_weight): {ratio:.2f}")
        print(f"Bias Init suggested: {bias_val:.4f}")

        return pos_weight_tensor, bias_val
    else:
        print("ALERT: There are no attacks on this dataset.")
        return None, None



# Auxiliary

In [10]:
ROOT_PATH = "./dataset_processed"

In [11]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


In [12]:
pos_weight_value, bias_value = check_class_balance(train_loader, "TRAIN")
print(f"pos_weight_value:{pos_weight_value}\n")
print(f"bias_value:{bias_value}\n")

#check_class_balance(val_loader, "VAL")

--- Analyzing TRAIN ---
Benign: 992229
Attack : 49564
Ratio (pos_weight): 20.02
Bias Init suggested: -2.9967
pos_weight_value:tensor([20.0191])

bias_value:-2.9966891636638033



# Main

## First configuration

In [12]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_mlp_biasinit.csv")

In [13]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 10
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.001

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
DROPOUT = 0.2

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


Using device: cpu


In [14]:
model_config = {
    "model_name": "EdgeMLP",
    "type": "MLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": bias_value,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": 1.0,
        "learning_rate": LR
    }
}


In [16]:
mlp_model = EdgeMLP(**model_config['model_params']).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value]).to(DEVICE))
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=LR)


best_aucpr = 0.0
best_wts = copy.deepcopy(mlp_model.state_dict())
best_metrics = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(mlp_model, train_loader, optimizer, criterion, DEVICE)
    metrics = evaluate(mlp_model, val_loader, criterion, DEVICE, prob_threshold=0.5)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val F1: {metrics['F1']:.4f} | Val Rec: {metrics['Recall']:.4f}")

    if metrics['AUC-PR'] > best_aucpr:
        best_aucpr = metrics['AUC-PR']
        best_metrics = metrics
        best_wts = copy.deepcopy(mlp_model.state_dict())


print(f"\nRestoring better weights (AUC-PR: {best_aucpr:.4f})...")
mlp_model.load_state_dict(best_wts)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=mlp_model)

print(f"\nBest AUC-PR for EdgeMLP obtained: {best_aucpr:.4f}")

--- Starting training: {'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 64, 'dropout': 0.2, 'output_bias_init': -2.9966891636638033} ---
Epoch 1 | Loss: 1.1226 | Val AUC-PR: 0.0711 | Val F1: 0.0764 | Val Rec: 1.0000
Epoch 2 | Loss: 1.0566 | Val AUC-PR: 0.0694 | Val F1: 0.0880 | Val Rec: 0.9964
Epoch 3 | Loss: 0.9902 | Val AUC-PR: 0.0661 | Val F1: 0.1305 | Val Rec: 0.8117
Epoch 4 | Loss: 0.9482 | Val AUC-PR: 0.0739 | Val F1: 0.1419 | Val Rec: 0.7952
Epoch 5 | Loss: 0.9257 | Val AUC-PR: 0.0758 | Val F1: 0.1526 | Val Rec: 0.7852
Epoch 6 | Loss: 0.8889 | Val AUC-PR: 0.0747 | Val F1: 0.1570 | Val Rec: 0.7754
Epoch 7 | Loss: 0.8746 | Val AUC-PR: 0.0757 | Val F1: 0.1578 | Val Rec: 0.7711
Epoch 8 | Loss: 0.8688 | Val AUC-PR: 0.0760 | Val F1: 0.1549 | Val Rec: 0.7788
Epoch 9 | Loss: 0.8676 | Val AUC-PR: 0.0768 | Val F1: 0.1576 | Val Rec: 0.7708
Epoch 10 | Loss: 0.8652 | Val AUC-PR: 0.0847 | Val F1: 0.1566 | Val Rec: 0.7757

Restoring better weights (AUC-PR: 0.0847)...
Experiment recorded in ./log

## Search grid

In [20]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 10
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5


search_space = {
    'pos_weight': [1.0, 2.0, 5.0],
    'hidden_dim': [32, 64]
}


In [21]:
model_config = {
    "model_name": None,
    "type": "MLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": None,
        "learning_rate": LR
    }
}

In [23]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0
total_exps = len(search_space['pos_weight']) * len(search_space['hidden_dim'])

for pw in search_space['pos_weight']:
    for h_dim in search_space['hidden_dim']:
        count += 1

        # Name
        exp_id = f"EdgeMLP_W{int(pw)}_H{h_dim}"
        print(f"\n{exp_id}")

        if exp_id in existing_experiments:
            print(f"[{count}/{total_exps}] Skipping {exp_id} (Already registered in CSV)")
            continue

        print(f"\n[{count}/{total_exps}] Starting: {exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        # Update model_config
        model_config['model_name'] = exp_id
        model_config["model_params"]["hidden_dim"] = h_dim
        model_config["extra_params"]["pos_weight"] = pw

        try:
            # Instantiate
            model = EdgeMLP(**model_config['model_params']).to(DEVICE)

            # Configure training
            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw]).to(DEVICE))

            # Deepcopy vars
            best_aucpr = 0.0
            best_wts = copy.deepcopy(model.state_dict())
            best_metrics = {}

            for epoch in range(EPOCHS):
                loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
                metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

                if metrics['AUC-PR'] > best_aucpr:
                    best_aucpr = metrics['AUC-PR']
                    best_metrics = metrics
                    best_wts = copy.deepcopy(model.state_dict())

            # Restore and save
            model.load_state_dict(best_wts)
            exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
            print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")

        except Exception as e:
            print(f"   FATAL ERROR in {exp_id}: {str(e)}")
            continue

        # Clean memory
        del model
        del optimizer
        del criterion
        del best_wts
        gc.collect()
        torch.cuda.empty_cache()


History loaded. 1 experiments already performed

EdgeMLP_W1_H32

[1/6] Starting: EdgeMLP_W1_H32
   Ep 1 | Loss: 0.1215 | Val AUC-PR: 0.0734 | Val Rec: 0.0000
   Ep 2 | Loss: 0.1261 | Val AUC-PR: 0.0716 | Val Rec: 0.0000
   Ep 3 | Loss: 0.1246 | Val AUC-PR: 0.0708 | Val Rec: 0.0000
   Ep 4 | Loss: 0.1253 | Val AUC-PR: 0.0728 | Val Rec: 0.0000
   Ep 5 | Loss: 0.1212 | Val AUC-PR: 0.0728 | Val Rec: 0.0000
   Ep 6 | Loss: 0.1202 | Val AUC-PR: 0.0724 | Val Rec: 0.0000
   Ep 7 | Loss: 0.1185 | Val AUC-PR: 0.1641 | Val Rec: 0.0000
   Ep 8 | Loss: 0.1185 | Val AUC-PR: 0.1568 | Val Rec: 0.0000
   Ep 9 | Loss: 0.1182 | Val AUC-PR: 0.1642 | Val Rec: 0.0000
   Ep 10 | Loss: 0.1179 | Val AUC-PR: 0.1629 | Val Rec: 0.0000
Experiment recorded in ./logs/experiments_log_mlp_biasinit.csv
Saved model: EdgeMLP_W1_H32_20260128_1706_AUC-PR_0.1642
   Best AUC-PR: 0.1642 (FPR: 0.0000)

EdgeMLP_W1_H64

[2/6] Starting: EdgeMLP_W1_H64
   Ep 1 | Loss: 0.1211 | Val AUC-PR: 0.0307 | Val Rec: 0.0000
   Ep 2 | Loss: 0